# 04 — Memory
Give stateless LLM APIs a sense of conversation history.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.5)


## 1. ConversationBufferMemory — Full History

In [ ]:
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain

# Stores every turn — highest fidelity, highest token cost
memory = ConversationBufferMemory(return_messages=True)
chat   = ConversationChain(llm=llm, memory=memory, verbose=False)

r1 = chat.predict(input="My name is Arjun and I am studying Generative AI.")
print("Turn 1:", r1[:120])

r2 = chat.predict(input="What should I learn after LangChain?")
print("Turn 2:", r2[:120])

r3 = chat.predict(input="Can you remind me what my name is?")
print("Turn 3:", r3[:120])

# Inspect stored messages
print("\nStored messages:")
for m in memory.chat_memory.messages:
    print(f"  [{m.__class__.__name__}]: {m.content[:60]}")


## 2. ConversationBufferWindowMemory — Sliding Window

In [ ]:
from langchain.memory import ConversationBufferWindowMemory

# k=3 → only keeps the last 3 turns (controls token cost)
window_memory = ConversationBufferWindowMemory(k=3, return_messages=True)
chat_window   = ConversationChain(llm=llm, memory=window_memory, verbose=False)

for i, msg in enumerate([
    "I love Python.",
    "I also enjoy data science.",
    "And I am learning LangChain.",
    "What topics have I mentioned so far?"
], 1):
    response = chat_window.predict(input=msg)
    print(f"Turn {i}: {response[:100]}")


## 3. ConversationSummaryMemory — Compressed History

In [ ]:
from langchain.memory import ConversationSummaryMemory

# Uses the LLM to compress old turns into a running summary
# Best for long conversations where full history is too expensive
summary_memory = ConversationSummaryMemory(llm=llm, return_messages=True)
chat_summary   = ConversationChain(llm=llm, memory=summary_memory, verbose=False)

chat_summary.predict(input="I am building a document Q&A system.")
chat_summary.predict(input="I am using FAISS as my vector store.")
chat_summary.predict(input="I am embedding with OpenAI text-embedding-3-small.")

# The memory now holds a compressed summary, not raw turns
print("Summary:", summary_memory.load_memory_variables({})["history"])


---
**Key takeaway:** Choose memory strategy based on your tradeoff — Buffer for accuracy, Window for cost control, Summary for long conversations.